# Exp 42: Oversample 8 without downsampling

Card: [exp42_oversample_8_only.md](../../docs/neuroslope/experiments/exp42_oversample_8_only.md)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from training.config import ROOT
from training.cv import CVResult
from analysis.plots import pr_curves, save_figures, use_style

CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp42_oversample_8_only" / "resnet18.yaml"
EXP29 = CONFIG_DIR / "exp29_downsample_5fold" / "resnet18.yaml"
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp42_oversample_8_only"

# dataviz reference palette, light mode
SERIES = {"exp29": "#eb6834", "oversample 8": "#2a78d6"}
SPLITS = {"train": "#eb6834", "validation": "#2a78d6"}
FOLD_STYLE = {0: "-", 1: "--", 2: ":", 3: "-.", 4: (0, (5, 1, 1, 1, 1, 1))}
use_style()

## Training


In [ ]:
results = {"exp29": CVResult.load(EXP29)}
results["oversample 8"] = CVResult.load(CONFIG)
figures = {}

## Precision-recall curves


In [ ]:
fig = pr_curves(results, colors=SERIES)
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

## Train and validation curves


In [ ]:
curves = pd.concat(
    {name: r.history() for name, r in results.items()}, names=["model"]
).reset_index(level=0)

fig, axes = plt.subplots(
    2, len(results), figsize=(4.2 * len(results), 6.4), sharex=True, sharey="row"
)
for row, metric in enumerate(["loss", "mAP"]):
    for col, name in enumerate(results):
        ax = axes[row, col]
        part = curves[(curves["model"] == name) & (curves["metric"] == metric)]
        for (split, fold), line in part.groupby(["split", "fold"]):
            ax.plot(
                line["epoch"],
                line["value"],
                color=SPLITS[split],
                ls=FOLD_STYLE[fold],
                lw=1.5,
                label=f"{split}, fold {fold}",
            )
        ax.set(title=f"{name}: {metric}", xlabel="epoch" if row else None)
    axes[row, 0].set_ylabel(metric)
axes[0, 0].legend(loc="best")
fig.tight_layout()
figures[f"{PREFIX}_training_curves.png"] = fig
plt.show()

In [ ]:
save_figures(figures, FIGURES)